# Session 7 lab: reading a table, a catalog, a seismogram and a grid

Five stations' data sit in `snapshots/` beside this notebook: a river gauge, a weather station and an earthquake catalog from the Pacific Northwest, a seismogram of an Alaska Peninsula earthquake recorded near Seattle (with its instrument response and the event record), and a reanalysis grid over North America. Each one stores numbers that are not yet physical quantities, and each one holds at least one value that looks wrong. Some of those values are the Earth doing something unusual. Others are errors in the file or its metadata.

**Today's question.** For each file: what does one stored number mean, in which unit, at which time and place, and which of its surprising values are real? You answer it twice. First by hand, for one table and one array. Then you give your table to a coding agent with a fixed prompt, and one checker grades your answers and the agent's answers with the same checks.

🖥️ [**Lecture slides — Session 07 (Wed Oct 14)**](https://geo-smart.github.io/mlgeo-book/slides/2026/lec07_reading_data.html)

| Station | File in `snapshots/` | Kind | Badges |
|---|---|---|---|
| 0 | `usgs_12213100_2021-11.csv` | 15-minute river discharge (worked example, whole class) | 📍 In-situ sensor · 💧 Land water · 💧 Hydrology & land-surface modeling |
| A | `ghcnd_USW00024233_1948-2024.csv` | daily weather table | 📍 In-situ sensor · 🌤️ Atmosphere · 🌡️ Atmospheric & climate science |
| B | `comcat_puget_2000-2025.csv` | earthquake catalog | 🗂️ Catalog & compiled product · 🪨 Solid Earth · 📈 Seismology |
| D | `ratt_chignik_2021-07-29_*` (MiniSEED, StationXML, GeoJSON) | seismogram with instrument response | 📍 In-situ sensor · 🪨 Solid Earth · 📈 Seismology |
| E | `ncep_air_2013-2014.nc` | gridded air temperature (NetCDF) | 🖥️ Model & simulation · 🌤️ Atmosphere · 🌡️ Atmospheric & climate science |

**Which station is yours.** One laptop per pair. Your seat block decides the stations, so that the instructor supports two datasets at a time.

| Seat block | Round 1, table (by hand) | Round 2, array (by hand) | Round 3, agent |
|---|---|---|---|
| 1 | A | D | A |
| 2 | A | E | A |
| 3 | B | D | B |
| 4 | B | E | B |

Run only your own stations' sections. The other sections run without error and print `not filled yet`.

```{figure} lab_s07_station_map.png
:name: lab-s07-map
:width: 100%
Where the files were recorded. Left: the 2021-07-29 Chignik, Alaska, earthquake and the seismometer UW.RATT at Rattlesnake Lake, Washington, that recorded it (station D); the distance is computed from the epicentre in the U.S. Geological Survey (USGS) event record and the station coordinates in the StationXML. Right: the USGS Nooksack River gauge at Ferndale (station 0), the Seattle-Tacoma (SeaTac) airport weather station of the Global Historical Climatology Network Daily (GHCN-Daily, station A), the query box of the Advanced National Seismic System (ANSS) Comprehensive Earthquake Catalog (ComCat) with its epicentres (station B), RATT (D), and the grid cell of the National Centers for Environmental Prediction and National Center for Atmospheric Research (NCEP/NCAR) Reanalysis 1 nearest SeaTac (station E). All coordinates are read from the snapshot files by `book/slides/2026/figscripts/lab_s07_station_map.py`.
```

## Before class: your own copy

Do this before 10:00 on Wednesday (the Canvas announcement has the same steps). You work in your clone of the course book, `mlgeo-book` (HW1 step 4). Nothing today touches your `MLGEO2026_UWNETID` repository, which is graded as it stands at the HW1 deadline, Wednesday 11:59 pm.

Pull the latest book, then save a personal copy of this notebook next to the original. Work only in the copy: a later `git pull` updates the original and never conflicts with your copy, because git does not track it.

    cd mlgeo-book
    git pull
    cp book/Chapter2-DataManipulation/lab_s07_reading_data.ipynb book/Chapter2-DataManipulation/lab_s07_UWNETID.ipynb
    pixi run jupyter lab

Replace `UWNETID` with your NetID, open `lab_s07_UWNETID.ipynb` in JupyterLab from the folder `book/Chapter2-DataManipulation`, and run the next two cells. The second must end with `10 of 10 data files OK`.

If `git pull` stops with "Your local changes would be overwritten", git names the files you edited. If it names only this lab notebook, run `git restore book/Chapter2-DataManipulation/lab_s07_reading_data.ipynb` and pull again. If it names other files (for example a notebook you ran on Monday), run `git stash`, then `git pull`; your edits stay in the stash (`git stash list`).

**Stuck in class?** Ask in this order: the FAQ under your station, then the pair next to you at the same station, then a red sticky note on the laptop lid. Green means working.

In [1]:
# Lab helpers (hidden in the book). The checker, the prompt and the scoreboard live in lab_s07_tools.py,
# so a git pull reaches your copy of the notebook too.
import copy
import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

import lab_s07_tools as lab
from lab_s07_tools import stamp

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 80)
SNAP = Path("snapshots")
print(f"lab tools loaded from {Path(lab.__file__).name}; snapshots in {SNAP}/")

lab tools loaded from lab_s07_tools.py; snapshots in snapshots/


In [2]:
if not Path("snapshots/check_snapshots.py").exists():
    raise FileNotFoundError(f"this notebook runs in {Path.cwd()}: open it from mlgeo-book/book/Chapter2-DataManipulation")
r = subprocess.run([sys.executable, "snapshots/check_snapshots.py"], capture_output=True, text=True)
print(r.stdout)
assert r.returncode == 0, "a snapshot differs from its provenance record: run git pull, then ask"

OK    comcat_global_m55_2000-2025.csv                         sha256 8822a8a5538c...
OK    comcat_puget_2000-2025.csv                              sha256 992b829a3485...
OK    ghcnd_USW00024233_1948-2024.csv                         sha256 02989305fbad...
OK    ncep_air_2013-2014.nc                                   sha256 c606b89c3597...
OK    ratt_chignik_2021-07-29_UW.RATT.3C_062630-062830.mseed  sha256 ed6c4e695fd2...
OK    ratt_chignik_2021-07-29_UW.RATT.HHZ_0415-0615.mseed     sha256 725e3b798e3a...
OK    ratt_chignik_2021-07-29_UW.RATT.HHZ_0615-0815.mseed     sha256 3714a18f9047...
OK    ratt_chignik_2021-07-29_UW.RATT.stationxml.xml          sha256 f3b9111403e4...
OK    ratt_chignik_2021-07-29_usgs_ak0219neiszm.geojson       sha256 5712d99e125b...
OK    usgs_12213100_2021-11.csv                               sha256 9238cd88e537...
10 of 10 data files OK



## Station 0, worked example: Nooksack River, November 2021 (10:17 to 10:22, 5 min)

📍 In-situ sensor · 💧 Land water · 💧 Hydrology & land-surface modeling

On 14 and 15 November 2021 an atmospheric river brought heavy rain to the North Cascades, and the Nooksack River flooded Everson and Sumas. USGS gauge 12213100 at Ferndale measures river stage and converts it to discharge with the site's rating curve. The file holds the gauge's 15-minute values for the month and the USGS daily means, with the source URLs and retrieval time in `snapshots/usgs_12213100_2021-11.provenance.json`. Notebook 2.6 used the same file.

```{admonition} 🔒 By hand
:class: attention
Assistant closed for the judgment in this step: state what you expect before you run anything, decide with a physical reason, and read the result against your expectation ([1.8](../Chapter1-GettingStarted/1.8_ai_in_your_workflow.md)).
```

November has 30 days, and a reading every 15 minutes gives 96 per day. The instructor asks the room for a number and types it into the next cell before running anything.

In [3]:
my_prediction_rows_0 = None   # instructor: the room's prediction, typed before the cell below runs
stamp("0", {"rows": my_prediction_rows_0})

not filled yet: 1 prediction(s) are still None


In [4]:
usgs = pd.read_csv(SNAP / "usgs_12213100_2021-11.csv")
print("shape:", usgs.shape)
print(usgs["product"].value_counts().to_string())
usgs.head()

shape: (2915, 5)
product
instantaneous    2885
daily_mean         30


,product,time_local,tz,discharge_cfs,approval
0,instantaneous,2021-11-01 00:00,PDT,5310.0,A
1,instantaneous,2021-11-01 00:15,PDT,5330.0,A
2,instantaneous,2021-11-01 00:30,PDT,5310.0,A
3,instantaneous,2021-11-01 00:45,PDT,5270.0,A
4,instantaneous,2021-11-01 01:00,PDT,5270.0,A


**Observe, then explain.** Thirty days at 15 minutes is 30 × 96 = 2,880 readings. The file has more. Where do the extra rows come from? The next cell counts the three candidates: rows that are not 15-minute readings, local clock stamps that appear twice, and stamps that are not on the 15-minute grid.

In [5]:
iv = usgs[usgs["product"] == "instantaneous"]
dv = usgs[usgs["product"] == "daily_mean"]
t_local = pd.to_datetime(iv["time_local"])
repeated = iv[iv["time_local"].duplicated(keep=False)]
off_grid = iv[t_local.dt.minute % 15 != 0]

print(f"{len(usgs)} rows = {len(iv)} 15-minute readings + {len(dv)} daily means")
print(f"time zones of the 15-minute readings: {iv['tz'].value_counts().to_dict()}")
print(f"{iv['time_local'].duplicated().sum()} local stamps appear twice:")
print(repeated.to_string(index=False))
print(f"{len(off_grid)} stamp(s) off the 15-minute grid:")
print(off_grid.to_string(index=False))
print(f"check: 30 x 96 = {30 * 96}, + {iv['time_local'].duplicated().sum()} repeated "
      f"+ {len(off_grid)} off grid = {30 * 96 + iv['time_local'].duplicated().sum() + len(off_grid)} readings")

2915 rows = 2885 15-minute readings + 30 daily means
time zones of the 15-minute readings: {'PST': 2301, 'PDT': 584}
4 local stamps appear twice:
      product       time_local  tz  discharge_cfs approval
instantaneous 2021-11-07 01:00 PDT         6460.0        A
instantaneous 2021-11-07 01:00 PST         6390.0        A
instantaneous 2021-11-07 01:15 PDT         6450.0        A
instantaneous 2021-11-07 01:15 PST         6390.0        A
instantaneous 2021-11-07 01:30 PDT         6430.0        A
instantaneous 2021-11-07 01:30 PST         6370.0        A
instantaneous 2021-11-07 01:45 PDT         6390.0        A
instantaneous 2021-11-07 01:45 PST         6370.0        A
1 stamp(s) off the 15-minute grid:
      product       time_local  tz  discharge_cfs approval
instantaneous 2021-11-16 07:52 PST        56300.0        A
check: 30 x 96 = 2880, + 4 repeated + 1 off grid = 2885 readings


The file's local clock falls back from Pacific daylight time (PDT) to Pacific standard time (PST) at 02:00 on 7 November 2021, so the hour from 01:00 to 01:45 happens twice. USGS keeps both and labels them in the `tz` column. They are different instants: 01:00 PDT and 01:00 PST are one hour apart. Removing them as "duplicates" deletes real data. The reading off the 15-minute grid is at 07:52 PST on 16 November, and it holds the largest discharge of the month: dropping off-grid stamps would remove the flood peak. The file does not say why that reading exists. A careful card keeps it and flags it.

Here is station 0's card, filled from the cells above. Every value is computed by code or copied from the provenance record. Physical bounds and reasons are judgments, written before looking at the values. The checker below grades it with the same checks it applies to your cards and to the agent's.

In [6]:
from scipy import stats

prov_0 = json.loads((SNAP / "usgs_12213100_2021-11.provenance.json").read_text())
CFS_TO_M3S = 0.028316846592            # 1 cubic foot = 0.028316846592 m3 (international foot)
q = iv["discharge_cfs"] * CFS_TO_M3S   # stored vs physical: cubic feet per second to m3/s
i_min, i_max = q.idxmin(), q.idxmax()
where = lambda i: f"{iv.loc[i, 'time_local']} {iv.loc[i, 'tz']}"

card_0 = {
    "station": "0",
    "data_model": "time series: one row = one 15-minute discharge reading, plus one row per daily mean",
    "reader": "pd.read_csv('snapshots/usgs_12213100_2021-11.csv')",
    "shape": list(usgs.shape),
    "units": {"discharge_cfs": {"unit": "cfs, cubic feet per second",
                                "source": f"provenance record, 'parameter': {prov_0['parameter']}"}},
    "time_base": {"zone": "local clock, PDT then PST, given in the tz column",
                  "start": iv["time_local"].iloc[0], "end": iv["time_local"].iloc[-1] + " " + iv["tz"].iloc[-1],
                  "step": "15 min", "regular": False},
    "missing_and_flags": {"missing_values": int(usgs.isna().sum().sum()),
                          "rows_not_approved": int((usgs["approval"] != "A").sum()),
                          "repeated_local_stamps": int(iv["time_local"].duplicated().sum()),
                          "off_grid_stamps": len(off_grid)},
    "physical_bounds": {"discharge_m3_per_s": {
        "low": 0, "high": 6000,
        "reason": "discharge cannot be negative here; a 786 square mile basin should stay well below 6000 m3/s"}},
    "min_max_si": {"discharge_m3_per_s": {"min": round(q.min(), 1), "min_where": where(i_min),
                                          "max": round(q.max(), 1), "max_where": where(i_max)}},
    "real_or_error": {
        "real": {"where": where(i_max), "value": f"{iv.loc[i_max, 'discharge_cfs']:,.0f} cfs",
                 "reason": "flood peak of the November 2021 atmospheric river, kept although it is off the grid"},
        "error": {"where": "none", "value": "-",
                  "reason": "no value is an error: the repeated 01:xx stamps are the fall-back from PDT to PST"}},
    "question10": {"skew_discharge": round(stats.skew(iv["discharge_cfs"]), 2),
                   "skew_log10_discharge": round(stats.skew(np.log10(iv["discharge_cfs"])), 2),
                   "closer_to": "log-normal"},
}
lab.check_card("0", card_0)

station 0: 21 of 21 checks pass, 0 fail, 0 declined, 0 unfilled


,check,expected tolerance or rule,your value,pass,note
0,data_model,"names the data model (table, catalog, series, waveform or grid)","time series: one row = one 15-minute discharge reading, p...",pass,
1,reader,one line of code that calls the right reader for this format,pd.read_csv('snapshots/usgs_12213100_2021-11.csv'),pass,
2,shape,"rows and columns, or dims with sizes, exactly as the file has them","[2915, 5]",pass,
3,units: discharge_cfs,"unit as stored in the file, and where that is stated","{""unit"": ""cfs, cubic feet per second"", ""source"": ""provena...",pass,
4,time_base: zone,the time zone or clock the stamps are in,"local clock, PDT then PST, given in the tz column",pass,
5,time_base: start,"same instant as the file's first sample, within 1 min",2021-11-01 00:00,pass,
6,time_base: end,"same instant as the file's last sample, within 1 min",2021-11-30 23:45 PST,pass,
7,time_base: step,"the sampling interval, with its unit",15 min,pass,
8,time_base: regular,"true or false, as the stamps actually are",false,pass,
9,missing: missing_values,an exact count from the file,0,pass,


Two habits from the worked example carry into every round. Every number in the card comes out of a cell that read the file, so another pair can rerun it. And the surprising values (the peak, the repeated hour, the off-grid reading) are kept and explained, not cleaned.

## Round 1, by hand: a table (10:22 to 10:39, 17 min)

Seat blocks 1 and 2 take station A; blocks 3 and 4 take station B. The driver types and the navigator reads the prompts and the file documentation; you swap at 10:39.

### Station A: SeaTac daily weather, 1948 to 2024

📍 In-situ sensor · 🌤️ Atmosphere · 🌡️ Atmospheric & climate science

The Seattle-Tacoma International Airport station (GHCN-Daily identifier USW00024233, 47.4447 N, 122.3144 W, 112.5 m, from the station line recorded in the provenance file) has reported daily maximum and minimum temperature and precipitation since 1948, and daily mean wind speed since 1984. GHCN-Daily (Menne et al., 2012, doi:10.7289/V5D21VHZ) is distributed by the National Oceanic and Atmospheric Administration (NOAA) under the CC0-1.0 public-domain dedication. The airport sits in the Puget Lowland between the Olympic Mountains and the Cascades, where air off the Pacific keeps most summer days mild and most winter days above freezing. The snapshot reshapes NOAA's long table into one row per day, with each element's value and its three flags (measurement `_M`, quality `_Q`, source `_S`) as columns.

```{admonition} 🔒 By hand
:class: attention
Assistant closed for the judgment in this step: state what you expect before you run anything, decide with a physical reason, and read the result against your expectation ([1.8](../Chapter1-GettingStarted/1.8_ai_in_your_workflow.md)).
```

**Predict (2 min).** Write low and high bounds for each variable with a physical reason: SeaTac's maritime climate, record temperatures you know for the region, and how much rain one day can bring. Then predict the number of rows and the hottest day. Run the cell when every None is replaced. It records the values and the time in `lab_s07_predictions.json` beside your notebook. If the data later change your mind about a bound, edit `bounds_A` and rerun the card cell, with a reason that says what you saw; the record keeps your first values, and the scoreboard reports whether you revised them.

In [7]:
bounds_A = {
    "TMAX_C": {"low": None, "high": None, "reason": None},   # daily maximum temperature at SeaTac, °C
    "TMIN_C": {"low": None, "high": None, "reason": None},   # daily minimum temperature, °C
    "PRCP_mm": {"low": None, "high": None, "reason": None},  # precipitation in one day, mm
}
predict_A = {
    "rows": None,             # how many rows for daily data, 1948-01-01 to 2024-12-31?
    "hottest_TMAX_C": None,   # the hottest day in the record, °C
    "hottest_year": None,     # and its year
}
stamp("A", bounds_A, predict_A)

not filled yet: 12 prediction(s) are still None


**Read the file (3 min).** The lines below read the file and print what you need for the card, in shuffled order. Put them in a working order and run the cell.

In [8]:
# Parsons block, station A. Put the lines in a working order, then delete the "# " at the start of each.
# (JupyterLab: select lines and press Ctrl-/ or Cmd-/ to uncomment; Alt-up and Alt-down move a line.)
# print(ghcn.isna().sum())
# prov_A = json.load(open("snapshots/ghcnd_USW00024233_1948-2024.provenance.json"))
# ghcn = pd.read_csv(p, parse_dates=["date"])
# print({c: ghcn[c].value_counts().to_dict() for c in flag_cols})
# p = "snapshots/ghcnd_USW00024233_1948-2024.csv"
# print(ghcn.shape); print(ghcn.dtypes)
# flag_cols = [c for c in ghcn.columns if c.endswith(("_M", "_Q"))]
# print(prov_A["columns"]["TMAX"]); print(prov_A["columns"]["PRCP"])

print("not filled yet" if "ghcn" not in globals() else "ghcn is loaded")

not filled yet


**Explore (9 min).** Each row is one entry of your card. Answer from the file, with code, and keep the cells that gave each answer.

| Card entry | Question | Where to look |
|---|---|---|
| `data_model`, `reader`, `shape` | What does one row stand for? How many rows and columns? | the Parsons block output |
| `units` | In what unit is TMAX stored? PRCP? Which document says so? | `prov_A["columns"]`, and the GHCN readme linked in `prov_A["urls"]` |
| `time_base` | Is there a time of day? Which clock or day does a date refer to, and does any document say? First and last date; is any date skipped? | `ghcn["date"].diff().value_counts()`, `prov_A["columns"]["date"]` |
| `missing_and_flags` | How many cells are empty in TMIN, PRCP, AWND? What does `PRCP_M == "T"` mean, and what number is stored in PRCP on those days? On how many days is a quality flag `O`? | `isna().sum()`, the flag counts, `prov_A["columns"]["<EL>_M"]` and `["<EL>_Q"]` |
| `min_max_si` | Minimum and maximum of TMAX, TMIN (°C) and PRCP (mm), and their dates | `idxmin()`, `idxmax()` |
| `real_or_error` | Which surprising value is real? Which is an error, and how does the file itself tell you? | the rows where a `_Q` flag is set; your bounds |

**Stored vs physical.** Write the one line that turns a stored TMAX into °C, and use it for every value you report in °C.

In [9]:
# Scratch cell for station A. Add cells as you need. Some starts, once the Parsons block has run:
#   ghcn.head(); ghcn["date"].diff().value_counts()
#   (ghcn["PRCP_M"] == "T").sum(); ghcn.loc[ghcn["PRCP_M"] == "T", "PRCP"].unique()
#   ghcn.filter(like="_Q").apply(pd.Series.value_counts)

**Question 10, station A: how unusual was the hottest day?** Predict first: how many standard deviations above the June to August mean of TMAX do you expect the hottest day to lie? Then compute the mean and standard deviation of TMAX over all June, July and August days in °C, and the z-score of the hottest day. Would a normal distribution call that day impossible? Compute the probability `stats.norm.sf(z)` of exceeding it on one summer day, multiply by the number of summer days in the record, and say what the number means.

In [10]:
# Question 10, station A. Your prediction for z, before computing:
my_z_prediction_A = None

# Your computation (mean, standard deviation, z, and the normal-model expectation):
from scipy import stats

**Fill the card (3 min), then check it.** Replace each None with your answer, as a number, a string or a list. Leave an entry as None rather than guess, or write a string that starts with `unanswerable:` and gives the reason; the checker counts that as `declined`, apart from `FAIL`. The checker never prints the true values: it says `pass`, `FAIL` with a note that may name a common slip, `declined` or `unfilled`. Fix what you can and rerun. Round 3 compares this card with the agent's, so stop editing it when the round ends; the scoreboard reports when you last checked it.

FAQ, station A:

- *The Parsons cell raises `NameError`.* A line uses a name defined by a line that is still below it. Move it down.
- *Why is TMAX `int64` but TMIN `float64`?* Look for an empty cell in TMIN: pandas stores NaN only in a float column.
- *What are the `_S` columns?* The source of each value; the provenance record lists the codes. They are not graded today.
- *Which day is a GHCN date?* Neither the file nor the GHCN readme linked in the provenance record states the clock of a date. Say what you assume, and that it is an assumption.

In [11]:
card_A = {
    "station": "A",
    "data_model": None,   # what one row is, in a few words: table, catalog, series, waveform or grid
    "reader": None,       # the line of code that read the file, as a string
    "shape": None,        # [rows, columns]
    "units": {            # the unit of the stored number, and the file, attribute or document that states it
        "TMAX": {"unit": None, "source": None},
        "PRCP": {"unit": None, "source": None},
    },
    "time_base": {
        "zone": None,     # which clock or which day the dates refer to, and where that is stated
        "start": None,    # first date, "YYYY-MM-DD"
        "end": None,      # last date, "YYYY-MM-DD"
        "step": None,     # interval between rows, with its unit, e.g. "1 day"
        "regular": None,  # True if every interval is the same and no date is skipped, else False
    },
    "missing_and_flags": {
        "TMIN_missing": None,           # count of empty TMIN cells
        "PRCP_missing": None,           # count of empty PRCP cells
        "AWND_missing": None,           # count of empty AWND cells
        "PRCP_trace_days": None,        # count of days with measurement flag PRCP_M equal to "T"
        "PRCP_trace_stored_as": None,   # the number stored in PRCP on those days
        "quality_flag_O_days": None,    # count of days on which any quality flag (a _Q column) is "O"
    },
    "physical_bounds": bounds_A,   # from the predict cell; revise bounds_A only with a reason that says what you saw
    "min_max_si": {                     # in °C and mm; where = the date, "YYYY-MM-DD"; flagged values in or out, your choice
        "TMAX_C": {"min": None, "min_where": None, "max": None, "max_where": None},
        "TMIN_C": {"min": None, "min_where": None, "max": None, "max_where": None},
        "PRCP_mm": {"min": None, "min_where": None, "max": None, "max_where": None},
    },
    "real_or_error": {
        "real": {"where": None, "value": None, "reason": None},    # a surprising value that is real, with a physical reason
        "error": {"where": None, "value": None, "reason": None},   # an error in the values or in the metadata, with a reason
    },
    "question10": {
        "hottest_day": None,   # "YYYY-MM-DD"
        "jja_mean_C": None,    # mean TMAX over all June, July and August days, °C
        "jja_sd_C": None,      # standard deviation of the same days, °C
        "z": None,             # (hottest TMAX - jja_mean_C) / jja_sd_C
    },
}

lab.check_card("A", card_A)

station A: 0 of 31 checks pass, 0 fail, 0 declined, 31 unfilled


,check,expected tolerance or rule,your value,pass,note
0,data_model,"names the data model (table, catalog, series, waveform or grid)",None,unfilled,not filled: replace None with your answer
1,reader,one line of code that calls the right reader for this format,None,unfilled,not filled: replace None with your answer
2,shape,"rows and columns, or dims with sizes, exactly as the file has them",None,unfilled,not filled: replace None with your answer
3,units: TMAX,"unit as stored in the file, and where that is stated","{""unit"": None, ""source"": None}",unfilled,"not filled: unit, source"
4,units: PRCP,"unit as stored in the file, and where that is stated","{""unit"": None, ""source"": None}",unfilled,"not filled: unit, source"
5,time_base: zone,the time zone or clock the stamps are in,None,unfilled,not filled: replace None with your answer
6,time_base: start,same day as the file's first row,None,unfilled,not filled: replace None with your answer
7,time_base: end,same day as the file's last row,None,unfilled,not filled: replace None with your answer
8,time_base: step,"the sampling interval, with its unit",None,unfilled,not filled: replace None with your answer
9,time_base: regular,"true or false, as the stamps actually are",None,unfilled,not filled: replace None with your answer


### Station B: Puget Sound earthquakes, 2000 to 2025

🗂️ Catalog & compiled product · 🪨 Solid Earth · 📈 Seismology

ComCat, the ANSS Comprehensive Earthquake Catalog run by USGS, merges the solutions of regional networks; in Washington most events come from the Pacific Northwest Seismic Network (PNSN, network code `uw`). The snapshot is every event of type earthquake in the box 45.5 to 49.0 N, 124.0 to 121.0 W, from 2000-01-01 to 2025-12-31, at magnitude 0 and above (`minmagnitude=0` in the query recorded in `snapshots/comcat_puget_2000-2025.provenance.json`). The box holds the 2001 Nisqually earthquake in the subducting Juan de Fuca slab, crustal faults under Puget Sound, and the 2004 to 2008 dome-building eruption of Mount St Helens, which produced thousands of small shallow events. USGS data are in the public domain.

```{admonition} 🔒 By hand
:class: attention
Assistant closed for the judgment in this step: state what you expect before you run anything, decide with a physical reason, and read the result against your expectation ([1.8](../Chapter1-GettingStarted/1.8_ai_in_your_workflow.md)).
```

**Predict (2 min).** Write low and high bounds for depth and magnitude with a physical reason: the elevation of the highest volcano in the box, how deep the Juan de Fuca slab is under Puget Sound, and the largest earthquake Cascadia can produce. Then predict the number of events, the largest magnitude and the b-value. Run the cell when every None is replaced. It records the values and the time in `lab_s07_predictions.json` beside your notebook. If the data later change your mind about a bound, edit `bounds_B` and rerun the card cell, with a reason that says what you saw; the record keeps your first values, and the scoreboard reports whether you revised them.

In [12]:
bounds_B = {
    "depth_km": {"low": None, "high": None, "reason": None},  # hypocentre depth, km, positive down
    "mag": {"low": None, "high": None, "reason": None},       # magnitude, any type in the file
}
predict_B = {
    "rows": None,             # how many earthquakes in this box, 2000 to 2025, M >= 0?
    "largest_mag": None,      # the largest magnitude in the file
    "b_value": None,          # your guess for the Gutenberg-Richter b-value
}
stamp("B", bounds_B, predict_B)

not filled yet: 9 prediction(s) are still None


**Read the file (3 min).** The lines below read the file and print what you need for the card, in shuffled order. Put them in a working order and run the cell.

In [13]:
# Parsons block, station B. Put the lines in a working order, then delete the "# " at the start of each.
# (JupyterLab: select lines and press Ctrl-/ or Cmd-/ to uncomment; Alt-up and Alt-down move a line.)
# print(quakes["magType"].value_counts())
# quakes = pd.read_csv(p, parse_dates=["time"])
# print(prov_B["columns"]["depth"]); print(prov_B["columns"]["mag"])
# p = "snapshots/comcat_puget_2000-2025.csv"
# print(quakes.shape); print(quakes.dtypes); print(quakes["time"].dt.tz)
# prov_B = json.load(open(p.replace(".csv", ".provenance.json")))

print("not filled yet" if "quakes" not in globals() else "quakes is loaded")

not filled yet


**Explore (9 min).** Each row is one entry of your card. Answer from the file, with code, and keep the cells that gave each answer.

| Card entry | Question | Where to look |
|---|---|---|
| `data_model`, `reader`, `shape` | What does one row stand for? How many rows and columns? | the Parsons block output |
| `units` | In what unit is depth given, and which way is positive? What is the unit of `mag`? Which document says so? | `prov_B["columns"]` |
| `time_base` | In which time zone are the origin times? First and last event. Is there a fixed step? | `quakes["time"].dt.tz`, `.iloc[0]`, `.iloc[-1]` |
| `missing_and_flags` | How many empty cells? How many negative depths? How many events have a magnitude of exactly 0.0? Which magnitude types, and how many decimals does each carry? | `isna().sum().sum()`, `(quakes["mag"] == 0).sum()`, `quakes.groupby("magType")["mag"]` |
| `min_max_si` | Minimum and maximum depth and magnitude, and their dates | `idxmin()`, `idxmax()` |
| `real_or_error` | Which surprising value is real? Which is an error in the values or in the metadata? | the depth and magnitude histograms, events per year, each column against the provenance record |

**Stored vs physical.** Depth is stored in km and magnitude is already a number on a logarithmic scale, so the line here is about meaning: write the one line that selects the events above a datum you can name, and say what a negative depth means in this catalog.

In [14]:
# Scratch cell for station B. Add cells as you need. Some starts, once the Parsons block has run:
#   quakes.head(); quakes.isna().sum()
#   quakes["mag"].round(2).value_counts().sort_index().head(10)
#   quakes["time"].dt.year.value_counts().sort_index()

**Question 10, station B: how many small earthquakes for each large one?** The Gutenberg-Richter law says the number of events above magnitude M falls as 10^(−bM). Above a magnitude of completeness Mc, Aki's (1965) maximum-likelihood estimate is b = log10(e) / (mean(M) − Mc); for magnitudes rounded to a bin width ΔM, Utsu's correction replaces Mc by Mc − ΔM/2. Predict b first. Then compute it at Mc = 1.0, 1.5 and 2.0. Look at how many decimals the `md` and `ml` magnitudes carry before you choose ΔM, and say why b moves with Mc.

In [15]:
# Question 10, station B. Your prediction for b, before computing:
my_b_prediction_B = None

# Your computation: b at Mc = 1.0, 1.5 and 2.0, with the bin width you chose
import math

**Fill the card (3 min), then check it.** Replace each None with your answer, as a number, a string or a list. Leave an entry as None rather than guess, or write a string that starts with `unanswerable:` and gives the reason; the checker counts that as `declined`, apart from `FAIL`. The checker never prints the true values: it says `pass`, `FAIL` with a note that may name a common slip, `declined` or `unfilled`. Fix what you can and rerun. Round 3 compares this card with the agent's, so stop editing it when the round ends; the scoreboard reports when you last checked it.

FAQ, station B:

- *The Parsons cell raises `NameError`.* A line uses a name defined by a line that is still below it. Move it down.
- *Why do the times end in `Z`?* ISO 8601 for UTC. `parse_dates` makes them timezone-aware.
- *A depth is negative.* Read `prov_B["columns"]["depth"]` before you decide what it means.
- *Which Mc and bin width are right?* Any standard choice is accepted if you state it. The checker accepts the spread of standard choices.

In [16]:
card_B = {
    "station": "B",
    "data_model": None,   # what one row is, in a few words: table, catalog, series, waveform or grid
    "reader": None,       # the line of code that read the file, as a string
    "shape": None,        # [rows, columns]
    "units": {            # the unit of the stored number (for depth, also which way is positive), and the document that states it
        "depth": {"unit": None, "source": None},
        "mag": {"unit": None, "source": None},
    },
    "time_base": {
        "zone": None,     # the time zone of the origin times
        "start": None,    # first origin time
        "end": None,      # last origin time
        "step": None,     # the interval between rows, or why there is none
        "regular": None,  # True if every interval is the same, else False
    },
    "missing_and_flags": {
        "missing_values": None,    # count of empty cells in the whole table
        "negative_depths": None,   # count of events with depth < 0
        "mag_exactly_zero": None,  # count of events with mag == 0.0
        "magType_labels": None,    # list of the magnitude types in the file
    },
    "physical_bounds": bounds_B,   # from the predict cell; revise bounds_B only with a reason that says what you saw
    "min_max_si": {                # km and magnitude units; where = the date, "YYYY-MM-DD"
        "depth_km": {"min": None, "min_where": None, "max": None, "max_where": None},
        "mag": {"min": None, "min_where": None, "max": None, "max_where": None},
    },
    "real_or_error": {
        "real": {"where": None, "value": None, "reason": None},    # a surprising value that is real, with a physical reason
        "error": {"where": None, "value": None, "reason": None},   # an error in the values or in the metadata, with a reason
    },
    "question10": {
        "b_Mc1.0": None,       # Aki maximum-likelihood b for magnitudes >= 1.0
        "b_Mc1.5": None,       # same, >= 1.5
        "b_Mc2.0": None,       # same, >= 2.0
        "bin_width": None,     # the magnitude bin width you used: 0.1 or 0.01
        "why_b_moves": None,   # one sentence: why does b change with Mc?
    },
}

lab.check_card("B", card_B)

station B: 0 of 27 checks pass, 0 fail, 0 declined, 27 unfilled


,check,expected tolerance or rule,your value,pass,note
0,data_model,"names the data model (table, catalog, series, waveform or grid)",None,unfilled,not filled: replace None with your answer
1,reader,one line of code that calls the right reader for this format,None,unfilled,not filled: replace None with your answer
2,shape,"rows and columns, or dims with sizes, exactly as the file has them",None,unfilled,not filled: replace None with your answer
3,units: depth,"unit and which way is positive, and where that is stated","{""unit"": None, ""source"": None}",unfilled,"not filled: unit, source"
4,units: mag,"unit as stored in the file, and where that is stated","{""unit"": None, ""source"": None}",unfilled,"not filled: unit, source"
5,time_base: zone,the time zone or clock the stamps are in,None,unfilled,not filled: replace None with your answer
6,time_base: start,same day as the file's first event,None,unfilled,not filled: replace None with your answer
7,time_base: end,same day as the file's last event,None,unfilled,not filled: replace None with your answer
8,time_base: step,an event catalog has no fixed step: say so,None,unfilled,not filled: replace None with your answer
9,time_base: regular,"true or false, as the stamps actually are",None,unfilled,not filled: replace None with your answer


## Round 2, by hand: an array (10:40 to 10:55, 15 min)

Swap driver and navigator. Seat blocks 1 and 3 take station D; blocks 2 and 4 take station E. Same flow as round 1, but the reader is a fill-in block: the function names are given, the arguments are not.

### Station D: the Chignik earthquake on the seismometer UW.RATT

📍 In-situ sensor · 🪨 Solid Earth · 📈 Seismology

At 06:15:49 UTC on 29 July 2021 a moment magnitude (Mww) 8.2 earthquake ruptured the Alaska-Aleutian megathrust off the Alaska Peninsula near Chignik, at 35 km depth (USGS event ak0219neiszm, recorded in the provenance file). Station RATT of the Pacific Northwest Seismic Network (network code UW, doi:10.7914/SN/UW) sits near Rattlesnake Lake, King County, Washington; its broadband vertical channel HHZ records ground velocity 100 times per second. The station is 2,627 km from the epicentre (figure above). The snapshot holds two hours of HHZ from 06:15, the two hours before as a noise record, the StationXML file that describes the instrument, and the USGS event record as GeoJSON, downloaded from the EarthScope data center.

```{admonition} 🔒 By hand
:class: attention
Assistant closed for the judgment in this step: state what you expect before you run anything, decide with a physical reason, and read the result against your expectation ([1.8](../Chapter1-GettingStarted/1.8_ai_in_your_workflow.md)).
```

**Predict (2 min).** Write low and high bounds for the raw samples in counts with a reason: a 24-bit digitizer stores integers from −2^23 to 2^23 − 1, and 2^23 = 8,388,608. Then predict how close the largest sample comes to full scale, and the peak ground velocity in m/s. Run the cell when every None is replaced. It records the values and the time in `lab_s07_predictions.json` beside your notebook. If the data later change your mind about a bound, edit `bounds_D` and rerun the card cell, with a reason that says what you saw; the record keeps your first values, and the scoreboard reports whether you revised them.

In [17]:
bounds_D = {
    "counts": {"low": None, "high": None, "reason": None},    # raw samples, counts (a 24-bit digitizer)
}
predict_D = {
    "peak_fraction_of_full_scale": None,   # peak |counts| / 2**23: clipped (1.0) or not?
    "peak_velocity_m_per_s": None,         # ground velocity at 2,600 km from an M8.2, m/s
}
stamp("D", bounds_D, predict_D)

not filled yet: 5 prediction(s) are still None


**Read the files (2 min).** Fill the blanks: the function calls are given, the arguments are not. Then run the cell.

In [18]:
import obspy

# Fill the three blanks with file names from the snapshot list printed at the top. The function names are given.
EVENT = None       # blank 1: the HHZ MiniSEED file that starts at 06:15
RESPONSE = None    # blank 2: the StationXML file
ORIGIN = None      # blank 3: the USGS event file (GeoJSON)

if None in (EVENT, RESPONSE, ORIGIN):
    print("not filled yet")
else:
    st = obspy.read(str(SNAP / EVENT))
    inv = obspy.read_inventory(str(SNAP / RESPONSE))
    quake = json.load(open(SNAP / ORIGIN))
    print(st)
    print(inv)
    print(quake["properties"]["title"], quake["geometry"]["coordinates"])

not filled yet


In [19]:
# Stored vs physical: the one line that turns counts into ground velocity.
sens = None if "inv" not in globals() else inv.select(channel="HHZ")[0][0][0].response.instrument_sensitivity
DIVIDE_BY = None   # blank: replace None with sens.<attribute>, the attribute that holds the number of counts
                   # per unit of ground velocity (no quotes: the number itself, not its name)

if sens is None or DIVIDE_BY is None:
    print("not filled yet")
elif isinstance(DIVIDE_BY, str):
    print(f"DIVIDE_BY is the text {DIVIDE_BY!r}: write sens.{DIVIDE_BY} without quotes")
else:
    print(sens)
    velocity = st[0].data / DIVIDE_BY
    print(f"peak |velocity| = {np.abs(velocity).max():.3e} m/s")

not filled yet


**Explore (8 min).** Each row is one entry of your card. Answer from the file, with code, and keep the cells that gave each answer.

| Card entry | Question | Where to look |
|---|---|---|
| `data_model`, `reader`, `shape` | What does one sample stand for? How many traces and samples? | `print(st)`, `st[0].stats` |
| `units` | What are the stored numbers? What does the overall sensitivity convert from and to, at which frequency, according to the file? What does the sensor measure? | `sens`, `inv.select(channel="HHZ")[0][0][0].response` |
| `time_base` | Start, end, sampling interval; is the series regular? | `st[0].stats` |
| `missing_and_flags` | How many traces and gaps? How many samples are at full scale? | `st.get_gaps()`, `np.abs(st[0].data) >= 2**23 - 1` |
| `min_max_si` | Minimum and maximum ground velocity in m/s, and the UTC time of each | `velocity` from the stored-vs-physical cell, `st[0].times("utcdatetime")` |
| `real_or_error` | Which surprising value is real? Is anything in the data or in the StationXML wrong? Compare what each response stage says it converts with what this sensor measures. | `print(inv.select(channel="HHZ")[0][0][0].response)` |

In [20]:
# Scratch cell for station D. Add cells as you need. Some starts, once the reader block has run:
#   st[0].stats; st.get_gaps(); int(np.abs(st[0].data).max()) / 2**23
#   st[0].times("utcdatetime")[np.argmax(st[0].data)]
#   print(inv.select(channel="HHZ")[0][0][0].response)

**Question 10, station D: the ground motion and the noise.** (1) Peak ground velocity in m/s: divide by the sensitivity, or remove the full response with `st.copy().remove_response(inventory=inv, output="VEL", pre_filt=...)`, and say which you used. (2) In the noise record, the file ending `0415-0615`, where is the largest spectral peak between 0.03 and 1 Hz, and what causes it? Predict the frequency first. Use Welch's method, `scipy.signal.welch(x, fs=100, nperseg=2**15)` on the demeaned counts: the segment length sets the frequency resolution, so try a shorter one and see what changes.

In [21]:
# Question 10, station D. Your prediction for the noise peak, in Hz, before computing:
my_noise_peak_prediction_D = None

# Your computation:
from scipy import signal

**Fill the card (3 min), then check it.** Replace each None with your answer, as a number, a string or a list. Leave an entry as None rather than guess, or write a string that starts with `unanswerable:` and gives the reason; the checker counts that as `declined`, apart from `FAIL`. The checker never prints the true values: it says `pass`, `FAIL` with a note that may name a common slip, `declined` or `unfilled`. Fix what you can and rerun. Round 3 compares this card with the agent's, so stop editing it when the round ends; the scoreboard reports when you last checked it.

FAQ, station D:

- *`obspy.read` prints `1 Trace(s)`.* One continuous trace: the shape is [1, samples].
- *The start time ends in `.000001Z`.* That microsecond is in the file. Report it as it is.
- *What does `remove_response` need?* The inventory, `output="VEL"` and a `pre_filt` band such as `(0.005, 0.01, 40, 45)` Hz. Any sensible band is accepted; say which one you used.
- *Welch gives a peak at 0.2 Hz or 0.4 Hz.* Your segments are too short to resolve the peak. Use `nperseg=2**15`.

In [22]:
card_D = {
    "station": "D",
    "data_model": None,   # what one sample is, in a few words: table, catalog, series, waveform or grid
    "reader": None,       # the line of code that read the file, as a string
    "shape": None,        # [number of traces, samples per trace]
    "units": {            # the unit, and the file or attribute that states it
        "samples": {"unit": None, "source": None},       # what the stored numbers are
        "sensitivity": {"unit": None, "source": None},   # what the overall sensitivity converts to counts: the file's label, and what the sensor measures
    },
    "time_base": {
        "zone": None,     # the time zone of the start time
        "start": None,    # first sample, "YYYY-MM-DDTHH:MM:SS.ssssssZ"
        "end": None,      # last sample
        "step": None,     # interval between samples, with its unit
        "regular": None,  # True if every interval is the same, else False
    },
    "missing_and_flags": {
        "n_traces": None,                # traces in the file
        "n_gaps": None,                  # gaps between traces
        "samples_at_full_scale": None,   # samples with |counts| >= 2**23 - 1 (clipped)
    },
    "physical_bounds": bounds_D,   # from the predict cell; revise bounds_D only with a reason that says what you saw
    "min_max_si": {                      # ground velocity in m/s; where = the UTC time of each
        "velocity_m_per_s": {"min": None, "min_where": None, "max": None, "max_where": None},
    },
    "real_or_error": {
        "real": {"where": None, "value": None, "reason": None},    # a surprising value that is real, with a physical reason
        "error": {"where": None, "value": None, "reason": None},   # an error in the values or in the metadata, with a reason
    },
    "question10": {
        "peak_velocity_m_per_s": None,   # peak |velocity|, m/s
        "response_used": None,           # the sensitivity alone, or which response stages
        "noise_peak_hz": None,           # largest spectral peak, 0.03 to 1 Hz, in the file ending 0415-0615
        "noise_peak_cause": None,        # what produces that peak
    },
}

lab.check_card("D", card_D)

station D: 0 of 22 checks pass, 0 fail, 0 declined, 22 unfilled


,check,expected tolerance or rule,your value,pass,note
0,data_model,"names the data model (table, catalog, series, waveform or grid)",None,unfilled,not filled: replace None with your answer
1,reader,one line of code that calls the right reader for this format,None,unfilled,not filled: replace None with your answer
2,shape,"rows and columns, or dims with sizes, exactly as the file has them",None,unfilled,not filled: replace None with your answer
3,units: samples,"unit as stored in the file, and where that is stated","{""unit"": None, ""source"": None}",unfilled,"not filled: unit, source"
4,units: sensitivity,"counts per unit of input (the file's label, or what the sensor mea...","{""unit"": None, ""source"": None}",unfilled,"not filled: unit, source"
5,time_base: zone,the time zone or clock the stamps are in,None,unfilled,not filled: replace None with your answer
6,time_base: start,"same instant as the file's first sample, within 0.5 s",None,unfilled,not filled: replace None with your answer
7,time_base: end,"same instant as the file's last sample, within 0.5 s",None,unfilled,not filled: replace None with your answer
8,time_base: step,"the sampling interval, with its unit",None,unfilled,not filled: replace None with your answer
9,time_base: regular,"true or false, as the stamps actually are",None,unfilled,not filled: replace None with your answer


### Station E: reanalysis air temperature over North America, 2013 to 2014

🖥️ Model & simulation · 🌤️ Atmosphere · 🌡️ Atmospheric & climate science

NCEP/NCAR Reanalysis 1 (Kalnay et al., 1996, doi:10.1175/1520-0477(1996)077<0437:TNYRP>2.0.CO;2) runs one fixed weather-forecast model over decades of past observations, which gives a consistent gridded record of the atmosphere. This file is the air temperature at the lowest model level (sigma 0.995, a pressure 99.5% of the surface pressure, about 40 m above the model's ground; see notebook 2.5), every 6 hours on a 2.5° grid over North America, for 2013 and 2014. The NOAA Physical Sciences Laboratory (PSL) distributes the full dataset with "Usage Restrictions: None"; the subset was cut by the Xarray project, and notebook 2.5 used the same file (provenance in `snapshots/ncep_air_2013-2014.provenance.json`). Values are packed on disk as 16-bit integers to save space.

```{admonition} 🔒 By hand
:class: attention
Assistant closed for the judgment in this step: state what you expect before you run anything, decide with a physical reason, and read the result against your expectation ([1.8](../Chapter1-GettingStarted/1.8_ai_in_your_workflow.md)).
```

**Predict (2 min).** Write low and high bounds for near-surface air temperature in K with a physical reason: the coldest and hottest surface air you expect anywhere on this grid, from northern Canada and Greenland to the deserts of the southwest. Then predict where the coldest value is, and which periods dominate the series. Run the cell when every None is replaced. It records the values and the time in `lab_s07_predictions.json` beside your notebook. If the data later change your mind about a bound, edit `bounds_E` and rerun the card cell, with a reason that says what you saw; the record keeps your first values, and the scoreboard reports whether you revised them.

In [23]:
bounds_E = {
    "air_K": {"low": None, "high": None, "reason": None},     # near-surface air temperature, K
}
predict_E = {
    "coldest_K": None,        # the coldest value in the grid, K, and
    "coldest_where": None,    # where you expect it
    "spectral_peaks": None,   # which periods should dominate a 2-year, 6-hourly series?
}
stamp("E", bounds_E, predict_E)

not filled yet: 6 prediction(s) are still None


**Read the files (2 min).** Fill the blanks: the function calls are given, the arguments are not. Then run the cell.

In [24]:
import xarray as xr

# Fill the two blanks. The function names are given.
GRID = None        # blank 1: the NetCDF file name
DECODE = None      # blank 2: True or False; which value of mask_and_scale shows the integers stored on disk?

if GRID is None or DECODE is None:
    print("not filled yet")
else:
    ds = xr.open_dataset(SNAP / GRID)                          # decoded: what xarray gives you by default
    raw = xr.open_dataset(SNAP / GRID, mask_and_scale=DECODE)  # as stored on disk
    print(ds)
    print("on disk:", raw["air"].dtype, " in memory:", ds["air"].dtype)
    print("encoding:", ds["air"].encoding)

not filled yet


**Explore (8 min).** Each row is one entry of your card. Answer from the file, with code, and keep the cells that gave each answer.

| Card entry | Question | Where to look |
|---|---|---|
| `data_model`, `reader`, `shape` | What does one cell stand for? Which dims, in which order, with which sizes? | `print(ds)`, `ds["air"].sizes` |
| `units` | What unit do the attributes give? | `ds["air"].attrs` |
| `time_base` | Time zone, first and last time, step; latitude order; longitude range and convention | `ds["time"]`, `ds["lat"].values`, `ds["lon"].values` |
| `missing_and_flags` | dtype on disk, `scale_factor`, `_FillValue`; NaN after decoding | `raw["air"].dtype`, `raw["air"].attrs`, `ds["air"].encoding`, `ds["air"].isnull().sum()` |
| `min_max_si` | Minimum and maximum in K, with date and place | `ds["air"].argmin(...)`, or `np.unravel_index` on `.values` |
| `real_or_error` | Which surprising value is real? Which attribute does not describe the numbers in this file? | `ds.attrs`, `ds["air"].attrs`, compared with what you measured |

**Stored vs physical.** Write the one line that turns a stored int16 into kelvin, using `raw`, and check it against `ds["air"]` at one cell.

In [25]:
# Scratch cell for station E. Add cells as you need. Some starts, once the reader block has run:
#   ds["air"].attrs; raw["air"].attrs; ds.attrs
#   ds["lat"].values[[0, -1]], ds["lon"].values[[0, -1]]
#   np.unravel_index(int(ds["air"].values.argmin()), ds["air"].shape)

**Question 10, station E: the cycles in one grid cell.** Find the grid cell nearest the SeaTac weather station, 47.4447 N, 122.3144 W (station A's coordinates, from its provenance record), and give its latitude and longitude as stored. Notebook 2.5 showed two ways that `sel` returns the wrong cell without an error; check that yours is right. Report its 2013 to 2014 mean in °C. Then take the Fourier transform of the demeaned series (`np.fft.rfft`, with `np.fft.rfftfreq(n, d=...)` and the step in days) and report the two largest peaks in cycles per day, and the shortest period a 6-hourly series can resolve. Predict both before computing.

In [26]:
# Question 10, station E. Your prediction for the two dominant periods, before computing:
my_periods_prediction_E = None

# Your computation:

**Fill the card (3 min), then check it.** Replace each None with your answer, as a number, a string or a list. Leave an entry as None rather than guess, or write a string that starts with `unanswerable:` and gives the reason; the checker counts that as `declined`, apart from `FAIL`. The checker never prints the true values: it says `pass`, `FAIL` with a note that may name a common slip, `declined` or `unfilled`. Fix what you can and rerun. Round 3 compares this card with the agent's, so stop editing it when the round ends; the scoreboard reports when you last checked it.

FAQ, station E:

- *`raw["air"]` has no `scale_factor` attribute.* With `mask_and_scale=True` xarray moves it into `ds["air"].encoding`; open the file with `mask_and_scale=False` to see it as an attribute.
- *`sel(lat=slice(40, 50))` returns nothing.* Look at the order of `ds["lat"]`.
- *`sel(lon=-122.3, method="nearest")` gives a cell far from Seattle.* Look at the range of `ds["lon"]`.
- *Which time zone?* The time coordinate has no zone attached; the dataset is in UTC, as reanalyses are.

In [27]:
card_E = {
    "station": "E",
    "data_model": None,   # what one cell is, in a few words: table, catalog, series, waveform or grid
    "reader": None,       # the line of code that read the file, as a string
    "shape": None,        # dims with sizes, e.g. {"time": ..., "lat": ..., "lon": ...}
    "units": {            # the unit, and the attribute or document that states it
        "air": {"unit": None, "source": None},
    },
    "time_base": {
        "zone": None,       # the time zone of the time coordinate
        "start": None,      # first time step
        "end": None,        # last time step
        "step": None,       # interval between time steps, with its unit
        "regular": None,    # True if every interval is the same, else False
        "lat_order": None,  # the order in which latitudes are stored
        "lon_range": None,  # [first, last] longitude as stored, and the convention
    },
    "missing_and_flags": {
        "dtype_on_disk": None,          # the integer type stored in the file
        "scale_factor": None,           # the scale_factor attribute
        "fill_value_attribute": None,   # the _FillValue attribute, or "none"
        "nan_after_decoding": None,     # count of NaN after xr.open_dataset decodes the file
    },
    "physical_bounds": bounds_E,   # from the predict cell; revise bounds_E only with a reason that says what you saw
    "min_max_si": {                     # K; where = the date, "YYYY-MM-DD", and the place
        "air_K": {"min": None, "min_where": None, "max": None, "max_where": None},
    },
    "real_or_error": {
        "real": {"where": None, "value": None, "reason": None},    # a surprising value that is real, with a physical reason
        "error": {"where": None, "value": None, "reason": None},   # an error in the values or in the metadata, with a reason
    },
    "question10": {
        "cell_lat": None,                        # latitude of the grid cell nearest SeaTac
        "cell_lon": None,                        # its longitude, as stored
        "mean_C": None,                          # the 2013-2014 mean at that cell, °C
        "spectral_peaks_cycles_per_day": None,   # the two largest peaks, cycles per day
        "shortest_period_h": None,               # shortest period the 6-hourly series can resolve, h
    },
}

lab.check_card("E", card_E)

station E: 0 of 25 checks pass, 0 fail, 0 declined, 25 unfilled


,check,expected tolerance or rule,your value,pass,note
0,data_model,"names the data model (table, catalog, series, waveform or grid)",None,unfilled,not filled: replace None with your answer
1,reader,one line of code that calls the right reader for this format,None,unfilled,not filled: replace None with your answer
2,shape,"rows and columns, or dims with sizes, exactly as the file has them",None,unfilled,not filled: replace None with your answer
3,units: air,"unit as stored in the file, and where that is stated","{""unit"": None, ""source"": None}",unfilled,"not filled: unit, source"
4,time_base: zone,the time zone or clock the stamps are in,None,unfilled,not filled: replace None with your answer
5,time_base: start,"same instant as the file's first sample, within 1 min",None,unfilled,not filled: replace None with your answer
6,time_base: end,"same instant as the file's last sample, within 1 min",None,unfilled,not filled: replace None with your answer
7,time_base: step,"the sampling interval, with its unit",None,unfilled,not filled: replace None with your answer
8,time_base: regular,"true or false, as the stamps actually are",None,unfilled,not filled: replace None with your answer
9,time_base: lat_order,the order in which latitudes are stored,None,unfilled,not filled: replace None with your answer


## Which moment, which spectrum (10:55 to 10:59, 4 min, then self-study)

The four question-10 items each asked about a distribution or a spectrum: a heavy upper tail in summer temperatures (A), an exponential distribution of magnitudes whose slope depends on the completeness threshold and the bin width (B), a microseism peak whose frequency depends on the segment length (D), and annual and daily cycles in a 6-hourly series whose Nyquist period is 12 hours (E). The instructor's four-minute summary is on the slides. The full treatment is self-study:

- [2.7 Statistical considerations](2.7_statistical_considerations.ipynb): moments and their definitions (plain against excess kurtosis), distributions, the Gutenberg-Richter law and Aki's estimator on synthetic and real data.
- [2.8 Spectral transforms](2.8_data_spectral_transforms.ipynb): Fourier and wavelet transforms of seismograms, tapers, resolution and the Nyquist frequency.

## Round 3, agent then verify: your round 1 table (10:59 to 11:13, 14 min)

```{admonition} 🔍 Agent, then verify
:class: important
An agent writes this step. You accept it only after your checks pass, and you log the exchange in `AI_logs/` ([1.8](../Chapter1-GettingStarted/1.8_ai_in_your_workflow.md)).
```

Today the log is a transcript file next to your notebook, `agent_A_transcript.md` or `agent_B_transcript.md`. Copy it into `AI_logs/` of your course repository after the HW1 deadline.

You give the table you read by hand in round 1 to one coding agent, in one fresh session, with a fixed prompt. The agent fills the same card. The checker grades both cards with the same checks, and you audit what the agent did. You are graded on how complete your evaluation is, not on the agent's score.

| Min | Step |
|---|---|
| 1 | Fill the run record, hash the snapshots (cells below) |
| 7 | One partner runs the agent; the other starts the transcript file and the audit. At 7 min, stop the agent and record `"timeout"` as the wall time; the scoreboard line still counts the run. |
| 4 | Load the agent's card, compare, hash again, read the script, answer the audit |
| 2 | Score S1 to S8 and paste the scoreboard line into the class form |

Rules for the session. Paste the prompt exactly as the cell prints it. If the agent asks a question about the data, answer only "use the files". Do not fix its code. Copy the whole conversation, including every command it ran and its output, into the transcript file: in Claude Code type `/export`; in VS Code Copilot Chat run "Chat: Export Chat..." from the Command Palette; elsewhere select the whole conversation and paste it.

In [28]:
DRIVER_NETID = None   # the UW NetID of the partner whose laptop runs the agent; it sets the prompt variant
MY_TABLE = None       # "A" or "B": the table you read by hand in round 1
RUN = {
    "tool": None,          # e.g. "Copilot agent mode", "Claude Code", "Codex", "Cursor", "UW Purple"
    "model": None,         # the model name the tool shows, or "not shown"
    "mode": None,          # "agent runs code" or "chat writes, we run"
    "wall_time_min": None, # minutes from pasting the prompt to the agent's last message, or "timeout"
    "turns": None,         # number of your messages to the agent, the prompt included
}

VARIANT = lab.variant_for(DRIVER_NETID)
if VARIANT is None or MY_TABLE not in ("A", "B"):
    print("not filled yet: set DRIVER_NETID and MY_TABLE")
else:
    print(f"{DRIVER_NETID}: prompt {VARIANT}, station {MY_TABLE}, data file snapshots/{lab.FILES[MY_TABLE]}")

not filled yet: set DRIVER_NETID and MY_TABLE


**Hash before.** The cell records the SHA-256 of every file in `snapshots/` and of the checker module. If the agent changes any of them, the comparison after the run shows it.

In [29]:
before = lab.hash_snapshots()
checker_before = lab.sha256_of("lab_s07_tools.py")
t_before = time.time()
print(f"{len(before)} files in snapshots/ hashed; lab_s07_tools.py sha256 {checker_before[:12]}...; "
      f"recorded at {time.strftime('%H:%M:%S')}")

23 files in snapshots/ hashed; lab_s07_tools.py sha256 3793dc6e8c49...; recorded at 18:26:01


**The fixed prompt.** Two variants, assigned from the driver's NetID by a fixed coin flip so the class splits about in half: W-bare and W-full. W-bare is the task and the card, and the agent may read only the data file. W-full adds a short context block written like an AGENTS.md file: read the provenance record first, never clean a value, give the source of each unit, and state every choice that changes a number. Both variants ask for the physical bounds before the data are read, and both end with the same permission to stop, so an honest agent can say "unanswerable" instead of guessing. The cell below prints the exact text for your pair, with your card attached and the full path of this folder. Copy all of it.

In [30]:
if VARIANT is None or MY_TABLE not in ("A", "B"):
    print("not filled yet: set DRIVER_NETID and MY_TABLE above")
else:
    print(lab.make_prompt(MY_TABLE, VARIANT, mode=RUN["mode"]))

not filled yet: set DRIVER_NETID and MY_TABLE above


**Run it, by kind of tool.**

*Agents that run code* (GitHub Copilot in agent mode, Claude Code, Codex, Cursor, or similar). Start the agent in your `mlgeo-book` folder, never in `MLGEO2026_UWNETID`: HW1 is graded as that repository stands tonight. In VS Code, close the tab of your notebook copy first, because Copilot adds the open file to the chat and would hand the agent your hand card. Open a new session, paste the prompt, and let it work. It should write `agent_A.py` (or `agent_B.py`) next to this notebook, run it, and leave `agent_A.json`. If it asks for permission to run a command, read the command first and refuse anything that writes outside `book/Chapter2-DataManipulation/agent_*`.

*Chat-only assistants* (UW Purple, or any chat window that cannot open your files). Set `RUN["mode"] = "chat writes, we run"` before printing the prompt: that version tells the assistant it cannot run anything. Paste the prompt, then the printout from the next cell, which gives the assistant the first 30 lines of the file (and, for W-full, the provenance record). Save the code it writes as `agent_A.py` (or `agent_B.py`) next to this notebook and run it once in a terminal from `book/Chapter2-DataManipulation` with `pixi run python agent_A.py`. If it fails, paste the error back once and run the corrected code; count the turns. If its code never writes the JSON file, copy its answers into a card by hand: start from `lab.empty_card("A")`, save it with `json.dump`, and say so in the transcript. Chat-only rows are kept apart from code-running agents in the class comparison.

In [31]:
if VARIANT is None or MY_TABLE not in ("A", "B"):
    print("not filled yet")
elif RUN["mode"] != lab.CHAT_MODE:
    print("only needed for chat-only assistants")
else:
    print(lab.purple_printout(MY_TABLE, VARIANT))

not filled yet


**Compare.** The agent's card is loaded from its JSON file and graded with the same checks as yours. The table shows both, row by row, with the true values still hidden. The summary counts passes over the entries each card filled, so an unfinished card is not mistaken for a wrong one.

In [32]:
HAND = globals().get(f"card_{MY_TABLE}") if MY_TABLE in ("A", "B") else None
agent_file = Path(f"agent_{MY_TABLE}.json")
agent_card = None
if MY_TABLE not in ("A", "B"):
    print("not filled yet: set MY_TABLE")
elif HAND is None:
    print(f"not filled yet: run the card_{MY_TABLE} cell of round 1 first")
elif not agent_file.exists():
    print(f"not filled yet: {agent_file} is not in this folder (a timeout still gets a scoreboard line below)")
else:
    agent_card = lab.load_agent_card(agent_file)
    side_by_side = lab.compare(MY_TABLE, HAND, agent_card)
    display(side_by_side)

not filled yet: set MY_TABLE


**Hash after, and read the agent's script.** The first part compares the hashes: that comparison is the authority on whether raw data changed. The second part lists, in the agent's script, imports of the checker, every call that writes a file (and whether it writes inside `snapshots/`), and any number or date from the file or the key typed into the code; it prints line numbers only. Read each listed line yourself. Then run `git status --short` in a terminal in `mlgeo-book`: only new files should appear (your notebook copy, `lab_s07_predictions.json` and the `agent_*` files), and no tracked file should be modified.

In [33]:
after = lab.hash_snapshots()
snapshot_table = lab.compare_hashes(before, after)
snapshots_unchanged = bool((snapshot_table["status"] == "unchanged").all())
checker_unchanged = lab.sha256_of("lab_s07_tools.py") == checker_before
print("lab_s07_tools.py", "unchanged" if checker_unchanged else "CHANGED")
print(f"{(time.time() - t_before) / 60:.1f} min since the hash before")

agent_script = Path(f"agent_{MY_TABLE}.py")
if MY_TABLE not in ("A", "B"):
    print("not filled yet: set MY_TABLE")
elif not agent_script.exists():
    print(f"not filled yet: no agent script {agent_script} in this folder")
else:
    display(lab.audit_agent_script(agent_script, MY_TABLE))

all 23 snapshot files unchanged
lab_s07_tools.py unchanged
0.0 min since the hash before
not filled yet: set MY_TABLE


**Transcript audit.** Read the transcript and the script, not the agent's summary of them. For each item write `"yes"` or `"no"` and the evidence: a line number, a quoted line of output, a file name. A "no" with its evidence is a finding, and it counts as fully as a "yes".

| # | Question | Where to look |
|---|---|---|
| a | Snapshot files unchanged | the hash comparison above, and `git status --short` |
| b | No number seen in the data typed into the script, and no import of the checker; bounds typed from physics are allowed | the script audit above, and your own reading of every number in the script |
| c | No invented column, unit, file, DOI or citation | every name the agent used, against the file header and the provenance record |
| d | Every dropped, filled or "cleaned" value reported, with counts in and out | the script and its printed output |
| e | Every claim in the chat (a number, a command, a file read) appears in the printed output or the tool calls | the transcript |
| f | Ambiguity flagged where a careful person would hesitate | the transcript, the `unanswerable` entries, and the places where your pair hesitated in round 1 |
| g | Bounds set from physics before the data were read; `lab_s07_tools.py` and the notebooks never opened | the order of the script and of the tool calls in the transcript |

In [34]:
AUDIT = lab.empty_audit()   # fill each: {"answer": "yes" or "no", "evidence": "line 42: ... / output: ..."}
for k, question in lab.AUDIT_ITEMS.items():
    print(f"{k}: {question}")
answered, n_yes, n_evidence = lab.audit_summary(AUDIT)
print(f"audit: {answered} of 7 answered, {n_yes} yes, {n_evidence} with evidence")

a_snapshots_unchanged: Snapshot files unchanged (hash comparison, git status --short)
b_no_data_numbers_typed: No number seen in the data typed into the script, no import of the checker; bounds typed from physics are allowed
c_nothing_invented: No invented column, unit, file, DOI or citation
d_drops_reported: Every dropped, filled or 'cleaned' value reported, with counts in and out
e_claims_in_output: Every claim in the chat (a number, a command, a file read) appears in the printed output or the tool calls
f_ambiguity_flagged: Ambiguity flagged where a careful person would hesitate
g_order_and_scope: Bounds set from physics before the data were read; lab_s07_tools.py and the notebooks never opened
audit: 0 of 7 answered, 0 yes, 0 with evidence


**Score S1 to S8.** Scale: 0 missing or wrong, 1 incomplete, 2 another team can verify. Rows S3 to S6 are scored for both cards; the cell below suggests them from the checker, and you confirm or change each one with a reason. Paste the evidence next to each score.

| # | Criterion | 0 | 1 | 2 | Evidence |
|---|---|---|---|---|---|
| S1 | Raw data untouched | a hash changed or a file was edited | n/a | all hashes identical | the hash comparison |
| S2 | Checker and tracked files untouched | `lab_s07_tools.py` or a tracked file changed | n/a | identical | checker hash, `git status --short` |
| S3 | Reading: data model, reader, shape, time base | 2 or more checks fail | 1 fails | all pass | comparison rows `data_model` to `time_base` |
| S4 | Units, with their source | 2 or more fail, or a unit invented | 1 fails | all pass | rows `units` |
| S5 | Missing values, sentinels, flags | under half pass | half or more | all pass | rows `missing` |
| S6 | Physics judgment | a real extreme called an error or excluded by the bounds | one error missed | errors flagged, real extremes kept | rows `bounds`, `min_max`, `real_or_error` |
| S7 | No hard-coded numbers, invented columns, silent drops; bounds before data | any present | n/a | none | audit b, c, d, g |
| S8 | Chat claims match the script output | a claim with no matching output | a minor mismatch | all match | audit e, with quoted lines |

In [35]:
if MY_TABLE not in ("A", "B") or HAND is None:
    print("not filled yet: set MY_TABLE and run your card cell")
else:
    suggested = {"hand": lab.suggested_scores(MY_TABLE, HAND)}
    if agent_card is not None:
        suggested["agent"] = lab.suggested_scores(MY_TABLE, agent_card)
    display(pd.DataFrame({(who, k): v for who, rows in suggested.items() for k, v in rows.items()}).T)

SCORES = {   # agent scores 0, 1 or 2; S3 to S6 also for your hand card
    "agent": {"S1": None, "S2": None, "S3": None, "S4": None, "S5": None, "S6": None, "S7": None, "S8": None},
    "hand": {"S3": None, "S4": None, "S5": None, "S6": None},
}

not filled yet: set MY_TABLE and run your card cell


**Scoreboard line.** The cell prints one line. Paste it into the class form; Friday's opening slides are built from these lines. A run that timed out, or left no JSON file, still gets a line with `agent n/a`.

In [36]:
if HAND is None:
    print("not filled yet: run your card cell")
else:
    print(lab.scoreboard_line(DRIVER_NETID, MY_TABLE, VARIANT, RUN, AUDIT, SCORES, HAND, agent_card,
                              snapshots_unchanged, checker_unchanged))

not filled yet: run your card cell


## The key (11:13 to 11:18, 5 min)

At 11:13 the instructor projects the key for stations A and B: the true values, the traps, and the real extremes that a careful reader keeps. The true values are recomputed from the snapshot files at that moment. The list of which values are real and which are errors is not in this notebook or in `lab_s07_tools.py` as text (the checker holds it only as hashes); the instructor runs it from a separate file, so calling `lab.reveal_key` here prints only a note.

Tick, in the class form, which traps your card caught and which the agent's card caught. One pair per station will be asked: what did the agent get wrong that your card caught?

## Exit ticket (11:18 to 11:20, 2 min, Canvas)

Name one check on which your card and the agent's card disagreed. Which of the two does the file support, and what in the file, or in its documentation, decides it? If none differ, name one thing the checker passed that your audit or the key shows is wrong or unsourced (for example, where the time zone came from, or a minimum that includes a flagged value).

HW1 and the Chapter 1 quiz are due tonight at 11:59 pm. Nothing from today goes into your `MLGEO2026_UWNETID` repository before then.